### Data Exploration

In [21]:
import pandas as pd

file_path = r"E:\Csci_3\subjects_AND_sampling_metadata_anonymized_full.csv"

# Candidate date columns
date_columns = ['created_at.users', 'sampling_date', 'clean...pool.date']

# Initialize dictionaries to track min/max dates
min_dates = {col: pd.Timestamp.max for col in date_columns}
max_dates = {col: pd.Timestamp.min for col in date_columns}

chunk_size = 100000

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False, usecols=date_columns):
    for col in date_columns:
        if col in chunk.columns:
            # Convert to datetime
            chunk[col] = pd.to_datetime(chunk[col], errors='coerce')
            
            # Normalize timezone: remove tz info if present
            chunk[col] = chunk[col].dt.tz_localize(None, ambiguous='NaT') if chunk[col].dt.tz else chunk[col]
            
            # Update min and max
            if not chunk[col].isna().all():
                min_dates[col] = min(min_dates[col], chunk[col].min())
                max_dates[col] = max(max_dates[col], chunk[col].max())

# Print results
for col in date_columns:
    print(f"{col}: Earliest = {min_dates[col]}, Latest = {max_dates[col]}")

# Overall timeline
overall_min = min(min_dates.values())
overall_max = max(max_dates.values())
print(f"Overall dataset timeline: {overall_min} → {overall_max}")

created_at.users: Earliest = 2007-02-21 21:44:51, Latest = 2024-10-28 18:00:46
sampling_date: Earliest = 2024-09-14 09:43:57, Latest = 2024-10-30 16:19:11
clean...pool.date: Earliest = 2024-09-19 15:18:00, Latest = 2024-10-30 16:19:11
Overall dataset timeline: 2007-02-21 21:44:51 → 2024-10-30 16:19:11


### Select Relevant Columns

In [22]:
cols_needed = [
    "clean...state_simple",
    "created_at.users",
    "tweets_historical",
    "entities.hashtags.subject_pool",
    "public_metrics.like_count",
    "public_metrics.retweet_count.tweets_historical"
]

### Build Clean Base Dataset

In [24]:
df_list = []

for chunk in pd.read_csv(file_path,
                         chunksize=chunk_size,
                         usecols=lambda x: x in cols_needed,
                         dtype=str,
                         low_memory=False):

    # Drop invalid rows
    chunk = chunk.dropna(subset=["created_at.users", "clean...state_simple"])

    # Convert date
    chunk["date"] = pd.to_datetime(chunk["created_at.users"], errors="coerce")

    # Convert numeric
    chunk["like_count"] = pd.to_numeric(chunk["public_metrics.like_count"], errors="coerce")
    chunk["retweet_count"] = pd.to_numeric(
        chunk["public_metrics.retweet_count.tweets_historical"], errors="coerce"
    )

    df_list.append(chunk)

df = pd.concat(df_list, ignore_index=True)

### Basic Exploration

In [25]:
print("Shape:", df.shape)
print("\nStates:", df["clean...state_simple"].nunique())
print("\nDate range:", df["date"].min(), "→", df["date"].max())

print("\nMissingness:")
print(df.isna().mean().sort_values(ascending=False))

Shape: (2147714, 9)

States: 50

Date range: 2007-02-21 21:44:51+00:00 → 2024-10-28 18:00:46+00:00

Missingness:
entities.hashtags.subject_pool                    0.91976
created_at.users                                  0.00000
public_metrics.retweet_count.tweets_historical    0.00000
public_metrics.like_count                         0.00000
tweets_historical                                 0.00000
clean...state_simple                              0.00000
date                                              0.00000
like_count                                        0.00000
retweet_count                                     0.00000
dtype: float64


### Focus on Swing States

In [26]:
swing_states = [
    "Arizona", "Georgia", "Michigan",
    "Nevada", "North Carolina",
    "Pennsylvania", "Wisconsin"
]

df = df[df["clean...state_simple"].isin(swing_states)]

### Construct Daily State-Level Panel

In [27]:
state_daily = df.groupby(["clean...state_simple", "date"]).agg({
    "like_count": "mean",
    "retweet_count": "mean",
    "tweets_historical": "count",
    "entities.hashtags.subject_pool": lambda x: x.notna().sum()
}).reset_index()

state_daily.rename(columns={
    "tweets_historical": "tweet_volume",
    "entities.hashtags.subject_pool": "hashtag_count"
}, inplace=True)

### Normalize Engagement

In [28]:
state_daily["engagement_score"] = (
    state_daily["like_count"].fillna(0) +
    state_daily["retweet_count"].fillna(0)
)

# Normalize by tweet volume
state_daily["engagement_per_tweet"] = (
    state_daily["engagement_score"] /
    state_daily["tweet_volume"].replace(0, np.nan)
)

### Detect Engagement Spikes

In [29]:
# Rolling baseline (7-day average)
state_daily["engagement_baseline"] = (
    state_daily.groupby("clean...state_simple")["engagement_per_tweet"]
    .transform(lambda x: x.rolling(7, min_periods=3).mean())
)

# Spike = % increase
state_daily["engagement_spike"] = (
    state_daily["engagement_per_tweet"] -
    state_daily["engagement_baseline"]
)

# Binary spike indicator
threshold = state_daily["engagement_spike"].quantile(0.75)

state_daily["high_spike"] = (state_daily["engagement_spike"] > threshold).astype(int)

### Define THE 5 Events

In [30]:
events = {
    "Event1": "2024-01-01",  # campaign start
    "Event2": "2024-06-01",  # debate
    "Event3": "2024-07-13",  # assassination attempt
    "Event4": "2024-07-21",  # Biden exits
    "Event5": "2024-10-20"   # final push
}

for key, date in events.items():
    state_daily[f"{key}_date"] = pd.to_datetime(date)

### Build Event Windows

In [31]:
state_daily["date"] = pd.to_datetime(state_daily["date"], utc=True)

for key in events.keys():
    
    # 1. Create event date
    state_daily[f"{key}_date"] = pd.to_datetime(events[key], utc=True)
    
    # 2. Compute days since event ✅ (THIS WAS MISSING)
    state_daily[f"{key}_days_since"] = (
        (state_daily["date"] - state_daily[f"{key}_date"]).dt.days
    )
    
    # 3. Create post indicator
    state_daily[f"{key}_post"] = (
        (state_daily[f"{key}_days_since"] > 0) &
        (state_daily[f"{key}_days_since"] <= 15)
    ).astype(int)

    # 4. Create pre indicator
    state_daily[f"{key}_pre"] = (
        (state_daily[f"{key}_days_since"] < 0) &
        (state_daily[f"{key}_days_since"] >= -15)
    ).astype(int)

### Define Treatment per Event

In [32]:
for key in events.keys():
    
    temp = state_daily[
        (state_daily[f"{key}_days_since"] >= 0) &
        (state_daily[f"{key}_days_since"] <= 3)
    ]

    spike_by_state = temp.groupby("clean...state_simple")["engagement_spike"].mean()

    threshold = spike_by_state.quantile(0.5)

    treated_states = spike_by_state[spike_by_state > threshold].index

    state_daily[f"{key}_treated"] = state_daily["clean...state_simple"].isin(treated_states).astype(int)

### Create DiD Interaction Terms

In [33]:
for key in events.keys():
    state_daily[f"{key}_interaction"] = (
        state_daily[f"{key}_treated"] *
        state_daily[f"{key}_post"]
    )

### Exploration Outputs

In [34]:
# Engagement by State

print(state_daily.groupby("clean...state_simple")["engagement_per_tweet"].mean())

# Spike Distribution

print(state_daily["high_spike"].value_counts(normalize=True))

# Event Coverage Check

for key in events.keys():
    print(key, state_daily[f"{key}_post"].sum())

clean...state_simple
Arizona           0.001253
Georgia           0.051225
Michigan          0.046154
Nevada            0.008718
North Carolina    0.006671
Pennsylvania      0.015109
Wisconsin         0.000881
Name: engagement_per_tweet, dtype: float64
high_spike
0    0.778761
1    0.221239
Name: proportion, dtype: float64
Event1 0
Event2 1
Event3 0
Event4 0
Event5 1
